<a href="https://colab.research.google.com/github/brinstussymethod/Credit-Card-Fraud-Detection-Machine-Learning-Project-/blob/main/ANN_CreditcardFraud_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"Created by Brian Andrade"
import kagglehub
from kagglehub import KaggleDatasetAdapter
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import GridSearchCV
import numpy as np



# Specify the name of the file within the dataset
dataset_file_name = "creditcard.csv"
df = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS,
                            "mlg-ulb/creditcardfraud",
                            dataset_file_name)

print("First 5 Records: ", df.head())

/tmp/ipykernel_9396/1803742645.py:12: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(KaggleDatasetAdapter.PANDAS,


Using Colab cache for faster access to the 'creditcardfraud' dataset.
First 5 Records:     Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ..

In [ ]:
!pip install scikeras

In [ ]:
!pip install --upgrade scikit-learn scikeras

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 116.3 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1


In [ ]:
# The class is the label while the rest a are the mysterious parameters causing a pattern we need to find using ANN. Let's isolate the label and the features
X = df.drop("Class", axis=1) # features
y = df["Class"] # labels

X.shape
y.shape

(284807,)

In [ ]:
from keras.models import Sequential
from keras.layers import Dense
from keras.metrics import Precision, Recall

def model_creator(optimizer = "adam", **kwargs):
    model = Sequential()
    model.add(Dense(12, input_shape=(30,), activation="relu"))
    model.add(Dense(8, activation="relu"))
    model.add(Dense(1, activation="sigmoid"))

    model.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=["accuracy", Precision(name="Precision"), Recall(name="Recall")]
    )

    return model

In [ ]:
seed = 2
np.random.seed(seed)
model = KerasClassifier(model=model_creator, verbose=2, meta={'_estimator_type': "classifier"})

In [ ]:
print("Shape of X (features):", X.shape)
print("Shape of y (labels):", y.shape)

Shape of X (features): (284807, 30)
Shape of y (labels): (284807,)


### Verifying for NaN and Infinite Values

In [ ]:
print("Checking for NaN values in X:")
print(X.isnull().sum().sum())

print("Checking for Infinite values in X:")
print(np.isinf(X).sum().sum())

Checking for NaN values in X:
0
Checking for Infinite values in X:
0


In [ ]:
print("Checking for NaN values in y:")
print(y.isnull().sum())

print("Checking for Infinite values in y:")
print(np.isinf(y).sum())

Checking for NaN values in y:
0
Checking for Infinite values in y:
0


In [ ]:
# Now to use GridSearch

param_grid = {
    "batch_size": [32, 64, 128],
    "epochs": [10, 20, 30]
}

grid = GridSearchCV(estimator=model, param_grid=param_grid, scoring = 'recall', cv=3)
grid_result = grid.fit(X, y)

print("Best Score: ", grid_result.best_score_)
print("Best Params: ", grid_result.best_params_)

Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


5934/5934 - 24s - 4ms/step - Precision: 0.0099 - Recall: 0.0109 - accuracy: 0.9970 - loss: 4.1278
Epoch 2/10
5934/5934 - 15s - 3ms/step - Precision: 0.0461 - Recall: 0.0509 - accuracy: 0.9971 - loss: 2.3300
Epoch 3/10
5934/5934 - 16s - 3ms/step - Precision: 0.0993 - Recall: 0.1018 - accuracy: 0.9974 - loss: 1.2003
Epoch 4/10
5934/5934 - 15s - 3ms/step - Precision: 0.0000e+00 - Recall: 0.0000e+00 - accuracy: 0.9983 - loss: 0.2257
Epoch 5/10
5934/5934 - 16s - 3ms/step - Precision: 0.0000e+00 - Recall: 0.0000e+00 - accuracy: 0.9986 - loss: 0.0129
Epoch 6/10
5934/5934 - 15s - 3ms/step - Precision: 0.0000e+00 - Recall: 0.0000e+00 - accuracy: 0.9986 - loss: 0.0195
Epoch 7/10
5934/5934 - 15s - 3ms/step - Precision: 0.0000e+00 - Recall: 0.0000e+00 - accuracy: 0.9986 - loss: 0.0109
Epoch 8/10
5934/5934 - 15s - 3ms/step - Precision: 0.0000e+00 - Recall: 0.0000e+00 - accuracy: 0.9986 - loss: 0.0109
Epoch 9/10
5934/5934 - 15s - 3ms/step - Precision: 0.0000e+00 - Recall: 0.0000e+00 - accuracy: 0.99

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


5934/5934 - 17s - 3ms/step - Precision: 0.0308 - Recall: 0.0295 - accuracy: 0.9966 - loss: 3.2015
Epoch 2/10
5934/5934 - 16s - 3ms/step - Precision: 0.0746 - Recall: 0.0737 - accuracy: 0.9967 - loss: 2.1318
Epoch 3/10
5934/5934 - 15s - 3ms/step - Precision: 0.0825 - Recall: 0.0944 - accuracy: 0.9965 - loss: 1.4095
Epoch 4/10
5934/5934 - 21s - 4ms/step - Precision: 0.1548 - Recall: 0.1475 - accuracy: 0.9970 - loss: 1.1307
Epoch 5/10
5934/5934 - 15s - 3ms/step - Precision: 0.1681 - Recall: 0.1740 - accuracy: 0.9970 - loss: 1.3176
Epoch 6/10
5934/5934 - 15s - 3ms/step - Precision: 0.2818 - Recall: 0.3068 - accuracy: 0.9974 - loss: 0.4468
Epoch 7/10
5934/5934 - 15s - 3ms/step - Precision: 0.5333 - Recall: 0.0708 - accuracy: 0.9982 - loss: 0.0515
Epoch 8/10
5934/5934 - 15s - 3ms/step - Precision: 0.7805 - Recall: 0.0944 - accuracy: 0.9983 - loss: 0.0138
Epoch 9/10
5934/5934 - 16s - 3ms/step - Precision: 0.6531 - Recall: 0.0944 - accuracy: 0.9983 - loss: 0.0123
Epoch 10/10
5934/5934 - 15s - 

...

1484/1484 - 7s - 4ms/step - Precision: 0.0034 - Recall: 0.0059 - accuracy: 0.9952 - loss: 2.4522
Epoch 2/30
1484/1484 - 4s - 2ms/step - Precision: 0.1797 - Recall: 0.1563 - accuracy: 0.9972 - loss: 1.3122
Epoch 3/30
1484/1484 - 4s - 3ms/step - Precision: 0.1945 - Recall: 0.1681 - accuracy: 0.9973 - loss: 1.4583
Epoch 4/30
1484/1484 - 4s - 3ms/step - Precision: 0.1699 - Recall: 0.1829 - accuracy: 0.9969 - loss: 1.5155
Epoch 5/30
1484/1484 - 4s - 2ms/step - Precision: 0.2632 - Recall: 0.2802 - accuracy: 0.9973 - loss: 0.9725
Epoch 6/30
1484/1484 - 4s - 2ms/step - Precision: 0.4325 - Recall: 0.5103 - accuracy: 0.9979 - loss: 0.3129
Epoch 7/30
1484/1484 - 4s - 3ms/step - Precision: 0.1792 - Recall: 0.1829 - accuracy: 0.9970 - loss: 1.4995
Epoch 8/30
1484/1484 - 4s - 2ms/step - Precision: 0.3325 - Recall: 0.3835 - accuracy: 0.9975 - loss: 0.7259
Epoch 9/30
1484/1484 - 4s - 2ms/step - Precision: 0.4574 - Recall: 0.4749 - accuracy: 0.9981 - loss: 0.3926
Epoch 10/30
1484/1484 - 4s - 3ms/step -

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1484/1484 - 6s - 4ms/step - Precision: 0.0022 - Recall: 0.0405 - accuracy: 0.9619 - loss: 293.6814
Epoch 2/30
1484/1484 - 4s - 3ms/step - Precision: 0.2069 - Recall: 0.3865 - accuracy: 0.9959 - loss: 0.6068
Epoch 3/30
1484/1484 - 4s - 2ms/step - Precision: 0.2840 - Recall: 0.3162 - accuracy: 0.9971 - loss: 1.1535
Epoch 4/30
1484/1484 - 4s - 2ms/step - Precision: 0.0570 - Recall: 0.0649 - accuracy: 0.9961 - loss: 3.3749
Epoch 5/30
1484/1484 - 4s - 3ms/step - Precision: 0.1912 - Recall: 0.2459 - accuracy: 0.9965 - loss: 1.6263
Epoch 6/30
1484/1484 - 4s - 3ms/step - Precision: 0.2745 - Recall: 0.3865 - accuracy: 0.9968 - loss: 1.6356
Epoch 7/30
1484/1484 - 4s - 2ms/step - Precision: 0.2830 - Recall: 0.4000 - accuracy: 0.9969 - loss: 0.7682
Epoch 8/30
1484/1484 - 4s - 3ms/step - Precision: 0.2566 - Recall: 0.2622 - accuracy: 0.9971 - loss: 1.8341
Epoch 9/30
1484/1484 - 4s - 3ms/step - Precision: 0.3665 - Recall: 0.5081 - accuracy: 0.9973 - loss: 0.9787
Epoch 10/30
1484/1484 - 4s - 2ms/step

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


2226/2226 - 8s - 4ms/step - Precision: 0.0728 - Recall: 0.1321 - accuracy: 0.9956 - loss: 3.3157
Epoch 2/10
2226/2226 - 5s - 2ms/step - Precision: 0.1328 - Recall: 0.1951 - accuracy: 0.9964 - loss: 2.2850
Epoch 3/10
2226/2226 - 6s - 3ms/step - Precision: 0.0923 - Recall: 0.0955 - accuracy: 0.9968 - loss: 3.9482
Epoch 4/10
2226/2226 - 5s - 2ms/step - Precision: 0.1577 - Recall: 0.2378 - accuracy: 0.9965 - loss: 2.4748
Epoch 5/10
2226/2226 - 6s - 3ms/step - Precision: 0.2431 - Recall: 0.4268 - accuracy: 0.9967 - loss: 1.4585
Epoch 6/10
2226/2226 - 5s - 2ms/step - Precision: 0.6505 - Recall: 0.1362 - accuracy: 0.9984 - loss: 1.2093
Epoch 7/10
2226/2226 - 6s - 3ms/step - Precision: 0.6933 - Recall: 0.1057 - accuracy: 0.9984 - loss: 0.0243
Epoch 8/10
2226/2226 - 5s - 2ms/step - Precision: 0.8158 - Recall: 0.0630 - accuracy: 0.9984 - loss: 0.0185
Epoch 9/10
2226/2226 - 6s - 3ms/step - Precision: 0.8649 - Recall: 0.0650 - accuracy: 0.9984 - loss: 0.0142
Epoch 10/10
2226/2226 - 6s - 2ms/step -